In [1]:
import os
import sys
import yaml
import pandas as pd
from pynxtools_em.examples.oasisb_utils import get_project_id

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.oasisb_utils import (
    APM_MIME_TYPES_SIDECAR,  # only for cross-referencing between apm and em collections
    APM_MIME_TYPES_SOLITARY,
    EM_HFIVE_MIME_TYPES_SIDECAR,  # hdf
    EM_HFIVE_MIME_TYPES_SOLITARY,
    EM_IMAGE_MIME_TYPES_SIDECAR,  # image
    EM_IMAGE_MIME_TYPES_SOLITARY,
    EM_KPY_MIME_TYPES_SIDECAR,  # kikuchipy diffraction pattern
    EM_KPY_MIME_TYPES_SOLITARY,
    EM_MIXED_MIME_TYPES_SIDECAR,  # mixed, spectrum, etc.
    EM_MIXED_MIME_TYPES_SOLITARY,
    EM_MTEX_MIME_TYPES_SIDECAR,  # mtex
    EM_MTEX_MIME_TYPES_SOLITARY,
    get_project_id,
    prepare_parsing,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

/home/kaiobach/Research/hu_hu_hu/sprint_fairmat1_final_pm/pynxtools-em/examples/oasisb
/media/kaiobach/berlin01/berlin01/MIA/TAICHI/Research/paper_paper_paper/scidat_nomad_em/bb_analysis/analysis/harvest_examples/data
/mnt/production/scidat_nomad_em/decompressed


## Decompress the original files from the scientists from the storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [2]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (1, 880)

with open(f"{src_directory}{os.sep}aaa_em_nomad_project_names.yaml") as fp:
    nomad_project_names: dict[str, str] = yaml.safe_load(fp)

# keep_searching_toggle = True
count: int = 0  # how many files to decompress
volume: int = 0  # how much byte volume does this add to scratch
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.project_name)
        if project_range[0] <= int(project_id) <= project_range[1]:
            if row.legal == "1" and row.use == "1":
                if project_id in nomad_project_names:
                    status = prepare_parsing(
                        f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                        src_directory,
                        project_id,
                        trg_directory,
                        report=True,
                        write=True,
                        mime_type="mixed",  # "image",  # "mtex", "hdf", "image", "mixed"
                        mime_type_solitary=EM_MIXED_MIME_TYPES_SOLITARY,
                        mime_type_sidecar=EM_MIXED_MIME_TYPES_SIDECAR,
                    )
                    for key, obj in status.items():
                        if obj["n"] > 0:
                            print(f"{project_id}, {key}, {obj['n']}, {obj['bytes']}")
                            count += obj["n"]
                            volume += obj["bytes"]
print(f"Batch queue completed, {count} files, {volume / 1024**3} GiB uncompressed")

049, .dm3, 67, 229158980
049, .emd, 12, 179894384
063, .dm3, 106, 1641551851
097, .bcf, 9, 985973601
116, .bcf, 2, 265560624
116, .dm3, 7, 96075099
117, .dm3, 10, 161244432
128, .emd, 1, 632621002
142, .dm3, 6, 23127118
142, .dm4, 30, 450447703
143, .ipj, 1, 134144
143, .dm3, 11, 3314988
143, .dm4, 59, 3995244534
160, .bcf, 9, 2611546584
163, .msa, 3, 275676
163, .dm4, 11, 7025969
167, .dm4, 14, 874788159
181, .dm4, 6, 103016149
191, .bcf, 1, 31035672
199, .emd, 9, 136674594
214, .dm3, 112, 1947229470
218, .bcf, 25, 2600541016
218, .dm3, 269, 4430342748
218, .dm4, 20, 326156716
231, .msa, 21, 579594
254, .dm3, 2, 34768538
257, .msa, 3, 182851
257, .dm3, 1, 217075411
277, .dm3, 7, 121690608
281, .bcf, 8, 61993856
286, .emd, 68, 4801027809
296, .dm4, 17, 69817563297
296, .emd, 6, 143650206
297, .dm4, 5, 23062936827
298, .dm4, 272, 57653062489
301, .emd, 53, 19611778910
303, .emd, 4, 9993760538
359, .ipj, 1, 5188608
360, .bcf, 126, 18540038608
361, .dm4, 263, 11578817154
369, .dm3, 2, 512

***

Programmatic identification of atom types from file names.

In [ ]:
pattern = os.path.join(trg_directory, f".mixed.decompressed.csv")
decompression_logfiles: list[str] = glob.glob(pattern)
for file in decompression_logfiles:
    print(file)